In [ ]:
# # IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# # RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
# import kagglehub
# kagglehub.login()


In [ ]:
# # IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# # THEN FEEL FREE TO DELETE THIS CELL.
# # NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# # ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# # NOTEBOOK.

# osbornnyakaru_afro_health_qa_path = kagglehub.dataset_download('osbornnyakaru/afro-health-qa')

# print('Data source import complete.')


In [ ]:
# %% [markdown]
# # AfroHealth QA — exp001 Full Pipeline (Part 1: Steps 1-4)
# **Platform**: Kaggle (T4 GPU) or Google Colab Free
# **Purpose**: Data audit, length bounds, held-out split, evaluation calibration

# %% — Cell 0: Install Dependencies
!pip install -q torch transformers accelerate bitsandbytes rouge-score scikit-learn pandas numpy tqdm sentencepiece protobuf fast-langdetect

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.0/790.0 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.1/296.1 kB 9.4 MB/s eta 0:00:00


In [ ]:
!pip install -q "bitsandbytes>=0.46.1"

In [ ]:
# %% — Cell 1: Configuration
from __future__ import annotations
import gc, hashlib, json, os, random, re, warnings
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ── Platform: set to "kaggle" or "colab" ──
PLATFORM = "colab"  # Change to "colab" if using Google Colab

# ── Paths ──
if PLATFORM == "kaggle":
    RAW_DIR = Path("/kaggle/input/datasets/osbornnyakaru/afro-health-qa")  # Your dataset name
    WORK_DIR = Path("/kaggle/working")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW_DIR = Path("/content/drive/MyDrive/afro-health-qa/data/raw")
    WORK_DIR = Path("/content/working")

PROCESSED_DIR = WORK_DIR / "data/processed"
SUBMISSIONS_DIR = WORK_DIR / "submissions"
EXPERIMENTS_DIR = WORK_DIR / "experiments"
CACHE_DIR = WORK_DIR / ".cache/afrolm_embeddings"
for d in [PROCESSED_DIR, SUBMISSIONS_DIR, EXPERIMENTS_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
TARGET_LEADERBOARD = 0.768095
EXPERIMENT_ID = "exp001"
HYPOTHESIS = "Zero-shot Aya-Expanse-8B lang-native prompts length-matched beam5 beats 0.768095"

# ── Seed in 5 places ──
def set_global_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        from transformers import set_seed; set_seed(seed)
    except Exception:
        pass

set_global_seed()
print(f"Seed={SEED} set in all 5 places.")

# ── GPU check ──
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")

Mounted at /content/drive
Seed=42 set in all 5 places.
CUDA available: True
VRAM: 15.6 GB


In [ ]:
# %% — Cell 2: Utility Functions
_GEEZ_RE = re.compile(r"[\u1200-\u137F]")

@dataclass(frozen=True)
class ColumnMapping:
    id_col: str
    question_col: str
    lang_col: str
    answer_col: str

@dataclass(frozen=True)
class SubmissionSchema:
    columns: tuple
    dtypes: dict
    n_rows: int
    ids_match_test_exactly: bool

def detect_column_mapping(train_df, val_df, test_df):
    common = set(train_df.columns) & set(val_df.columns) & set(test_df.columns)
    id_col = [c for c in common if c.lower() == "id" or c.lower().endswith("id")][0]
    question_col = [c for c in common if c.lower() in {"question","input","prompt","query"} and c != id_col][0]
    answer_col = [c for c in set(train_df.columns) & set(val_df.columns)
                  if c not in test_df.columns and c.lower() in {"response","answer","output","target"}][0]
    lang_candidates = [c for c in common if c not in {id_col, question_col} and c.lower() in {"language","lang","subset"}]
    if lang_candidates:
        lang_col = lang_candidates[0]
    else:
        leftovers = [c for c in common if c not in {id_col, question_col}]
        lang_col = leftovers[0]
    return ColumnMapping(id_col=id_col, question_col=question_col, lang_col=lang_col, answer_col=answer_col)

def infer_lang_code_map(values):
    mapping = {}
    for raw in sorted({str(v) for v in values}):
        key = raw.strip(); lower = key.lower()
        if lower.startswith("swa") or "swahili" in lower: mapping[key] = "swa"
        elif lower.startswith("lug") or "luganda" in lower: mapping[key] = "lug"
        elif lower.startswith("aka") or "twi" in lower or "akan" in lower: mapping[key] = "aka"
        elif lower.startswith("amh") or "amharic" in lower: mapping[key] = "amh"
        elif lower.startswith("eng") or "english" in lower: mapping[key] = "eng"
        else: raise ValueError(f"unknown language/subset code: {raw}")
    return mapping

def contains_geez(text): return bool(_GEEZ_RE.search(text or ""))

def canonical_prompt_language(lang_code): return "eng" if lang_code == "eng" else lang_code

def lang_distribution(df, lang_col):
    counts = df[lang_col].astype(str).value_counts(dropna=False).sort_index()
    pct = counts / len(df) * 100.0
    return pd.DataFrame({"count": counts, "pct": pct.round(2)})

def length_stats(df, lang_col, answer_col, lang_code_map):
    rows = []
    for raw_lang, group in df.groupby(lang_col, sort=True):
        canonical = lang_code_map[str(raw_lang)]
        tc = group[answer_col].astype(str).map(lambda x: len(x.split()))
        rows.append({"raw_lang": raw_lang, "lang": canonical, "n": len(group),
                      "mean": round(float(tc.mean()),2), "median": round(float(tc.median()),2),
                      "p10": int(np.percentile(tc,10)), "p90": int(np.percentile(tc,90))})
    return pd.DataFrame(rows).sort_values(["lang","raw_lang"]).reset_index(drop=True)

def build_submission_schema(sample_df, test_df, id_col):
    s_ids = sample_df.iloc[:,0].astype(str).tolist()
    t_ids = test_df[id_col].astype(str).tolist()
    return SubmissionSchema(columns=tuple(sample_df.columns.tolist()),
                            dtypes={c: str(d) for c,d in sample_df.dtypes.items()},
                            n_rows=len(sample_df), ids_match_test_exactly=s_ids==t_ids)

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print("Utility functions loaded.")

Utility functions loaded.


In [ ]:
# %% — Cell 3: STEP 1 — Ingest and Audit All Four Files
print("=" * 70)
print("STEP 1 — INGEST AND AUDIT ALL FOUR FILES")
print("=" * 70)

train = pd.read_csv(RAW_DIR / "Train.csv", dtype=str)
val   = pd.read_csv(RAW_DIR / "Val.csv", dtype=str)
test  = pd.read_csv(RAW_DIR / "Test.csv", dtype=str)
sample = pd.read_csv(RAW_DIR / "SampleSubmission.csv", dtype=str)

print(f"Train.csv     : {len(train)} rows × {len(train.columns)} cols  |  columns: {train.columns.tolist()}")
print(f"Val.csv       : {len(val)} rows × {len(val.columns)} cols  |  columns: {val.columns.tolist()}")
print(f"Test.csv      : {len(test)} rows × {len(test.columns)} cols  |  columns: {test.columns.tolist()}")
print(f"SampleSubmission.csv : {len(sample)} rows × {len(sample.columns)} cols  |  columns: {sample.columns.tolist()}")

# 1a. Column mapping
mapping = detect_column_mapping(train, val, test)
print(f"\n1a. Column mapping: {asdict(mapping)}")
id_col, question_col, lang_col, answer_col = mapping.id_col, mapping.question_col, mapping.lang_col, mapping.answer_col

# Language code map
all_lang_values = (train[lang_col].astype(str).tolist() + val[lang_col].astype(str).tolist()
                   + test[lang_col].astype(str).tolist())
LANG_CODE_MAP = infer_lang_code_map(all_lang_values)
print(f"\n1i. LANG_CODE_MAP = {json.dumps(LANG_CODE_MAP, indent=2, ensure_ascii=False)}")

# 1b. Language distribution
print("\n1b. Language distribution")
train_dist = lang_distribution(train, lang_col)
val_dist = lang_distribution(val, lang_col)
print("Train:\n", train_dist.to_string())
print("Val:\n", val_dist.to_string())
for rl in sorted(set(train_dist.index) | set(val_dist.index)):
    tp = float(train_dist.loc[rl,"pct"]) if rl in train_dist.index else 0.0
    vp = float(val_dist.loc[rl,"pct"]) if rl in val_dist.index else 0.0
    if tp < 10 or vp < 10: print(f"  [FLAG] {rl} below 10% in {'train' if tp<10 else 'val'}")
    if abs(tp-vp) > 15: print(f"  [FLAG] {rl} gap {abs(tp-vp):.1f}pp")

# 1c/1d. Answer length distributions
print("\n1c. Train answer length distribution:")
train_len = length_stats(train, lang_col, answer_col, LANG_CODE_MAP)
print(train_len.to_string(index=False))
print("\n1d. Val answer length distribution:")
val_len = length_stats(val, lang_col, answer_col, LANG_CODE_MAP)
print(val_len.to_string(index=False))

# 1e. Missing/null values
print("\n1e. Missing/null values")
for name, df in [("train",train),("val",val),("test",test),("sample",sample)]:
    nulls = df.isna().sum().to_dict()
    print(f"  {name}: {nulls}")

# Drop null answers if any
for name in ("train","val"):
    df_ref = train if name=="train" else val
    if df_ref[answer_col].isna().any():
        before = len(df_ref)
        df_ref = df_ref.dropna(subset=[answer_col]).reset_index(drop=True)
        print(f"  Dropped {before - len(df_ref)} null-answer rows from {name}")
        if name=="train": train = df_ref
        else: val = df_ref

assert not test[question_col].isna().any(), "CRITICAL: Test has null questions!"

# 1f. Amharic script check
print("\n1f. Amharic script check")
suspect_frames = []
for name, df in [("train",train),("val",val),("test",test)]:
    amh_mask = df[lang_col].astype(str).map(lambda x: LANG_CODE_MAP.get(str(x))=="amh")
    has_q = question_col in df.columns
    if has_q:
        suspect = df.loc[amh_mask & ~df[question_col].astype(str).map(contains_geez)].copy()
    else:
        suspect = pd.DataFrame()
    if len(suspect):
        suspect["source_file"] = name
        suspect_frames.append(suspect)
        df = df.loc[~(amh_mask & ~df[question_col].astype(str).map(contains_geez))].reset_index(drop=True)
        if name=="train": train = df
        elif name=="val": val = df
        else: test = df
    print(f"  {name}: {len(suspect)} suspect Amharic rows quarantined")

if suspect_frames:
    pd.concat(suspect_frames, ignore_index=True).to_csv(PROCESSED_DIR/"suspect_amharic.csv", index=False)

# 1g. Submission format fingerprint
print("\n1g. Submission format fingerprint")
SUBMISSION_SCHEMA = build_submission_schema(sample, test, id_col)
print(f"  Columns: {list(SUBMISSION_SCHEMA.columns)}")
print(f"  Dtypes: {SUBMISSION_SCHEMA.dtypes}")
print(f"  Rows: {SUBMISSION_SCHEMA.n_rows}")
print(f"  IDs match Test: {SUBMISSION_SCHEMA.ids_match_test_exactly}")

# 1h. Overlap check
print("\n1h. Train/Val/Test overlap check")
tq = set(train[question_col].astype(str))
vq = set(val[question_col].astype(str))
teq = set(test[question_col].astype(str))
print(f"  Train ∩ Val: {len(tq & vq)}")
print(f"  Train ∩ Test: {len(tq & teq)}")
print(f"  Val ∩ Test: {len(vq & teq)}")

# Save audit report
audit_lines = [
    "# Data Audit Report", "",
    "## File shapes",
    f"- Train: {len(train)} rows, cols={train.columns.tolist()}",
    f"- Val: {len(val)} rows, cols={val.columns.tolist()}",
    f"- Test: {len(test)} rows, cols={test.columns.tolist()}",
    f"- Sample: {len(sample)} rows, cols={sample.columns.tolist()}", "",
    f"## Column mapping\n{asdict(mapping)}", "",
    f"## Language code map\n```json\n{json.dumps(LANG_CODE_MAP,indent=2,ensure_ascii=False)}\n```", "",
    f"## Submission schema\n```json\n{json.dumps(asdict(SUBMISSION_SCHEMA),indent=2)}\n```", "",
    f"## Overlaps\n- Train∩Val: {len(tq&vq)}\n- Train∩Test: {len(tq&teq)}\n- Val∩Test: {len(vq&teq)}",
]
(PROCESSED_DIR / "data_audit_report.md").write_text("\n".join(audit_lines), encoding="utf-8")
print("\n✅ Step 1 complete. Audit saved to data/processed/data_audit_report.md")

STEP 1 — INGEST AND AUDIT ALL FOUR FILES
Train.csv     : 29815 rows × 4 cols  |  columns: ['ID', 'input', 'output', 'subset']
Val.csv       : 6686 rows × 4 cols  |  columns: ['ID', 'input', 'output', 'subset']
Test.csv      : 2618 rows × 3 cols  |  columns: ['ID', 'input', 'subset']
SampleSubmission.csv : 2618 rows × 4 cols  |  columns: ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']

1a. Column mapping: {'id_col': 'ID', 'question_col': 'input', 'lang_col': 'subset', 'answer_col': 'output'}

1i. LANG_CODE_MAP = {
  "Aka_Gha": "aka",
  "Amh_Eth": "amh",
  "Eng_Eth": "eng",
  "Eng_Gha": "eng",
  "Eng_Ken": "eng",
  "Eng_Uga": "eng",
  "Lug_Uga": "lug",
  "Swa_Ken": "swa"
}

1b. Language distribution
Train:
          count    pct
subset               
Aka_Gha   4455  14.94
Amh_Eth   1845   6.19
Eng_Eth   3915  13.13
Eng_Gha   4443  14.90
Eng_Ken   2080   6.98
Eng_Uga   7624  25.57
Lug_Uga   3383  11.35
Swa_Ken   2070   6.94
Val:
          count    pct
subset               
Aka_Gha   1114 

In [ ]:
# %% — Cell 4: STEP 2 — Build Per-Language Length Bounds
print("\n" + "=" * 70)
print("STEP 2 — BUILD PER-LANGUAGE LENGTH BOUNDS TABLE")
print("=" * 70)

LENGTH_BOUNDS = {}
comparison_rows = []
bound_notes = []

for lang in sorted(train_len["lang"].unique()):
    tg = train_len[train_len["lang"]==lang]
    vg = val_len[val_len["lang"]==lang]
    tp10 = int(tg["p10"].mean()); tp90 = int(tg["p90"].mean())
    vp10 = int(vg["p10"].mean()) if len(vg) else tp10
    vp90 = int(vg["p90"].mean()) if len(vg) else tp90

    if abs(vp90 - tp90) > 20:
        avg_p10 = int(round((tp10+vp10)/2)); avg_p90 = int(round((tp90+vp90)/2))
        chosen = {"min_new_tokens": max(1,int(avg_p10*0.8)), "max_new_tokens": avg_p90+30}
        bound_notes.append(f"{lang}: Train/Val p90 diverged by {abs(vp90-tp90)}, using average bounds.")
    else:
        chosen = {"min_new_tokens": max(1,int(tp10*0.8)), "max_new_tokens": tp90+30}

    LENGTH_BOUNDS[lang] = chosen
    comparison_rows.append({"lang":lang, "train_p10":tp10, "train_p90":tp90,
                            "val_p10":vp10, "val_p90":vp90,
                            "final_min":chosen["min_new_tokens"], "final_max":chosen["max_new_tokens"]})

print(pd.DataFrame(comparison_rows).to_string(index=False))
for n in bound_notes: print(f"  NOTE: {n}")
print(f"\nLENGTH_BOUNDS = {json.dumps(LENGTH_BOUNDS, indent=2, ensure_ascii=False)}")
print("✅ Step 2 complete.")


STEP 2 — BUILD PER-LANGUAGE LENGTH BOUNDS TABLE
lang  train_p10  train_p90  val_p10  val_p90  final_min  final_max
 aka         32        180       26      180         25        210
 amh         10         30       10       31          8         60
 eng         25        121       25      124         20        151
 lug         21        150       18      151         16        180
 swa         25        167       29      160         20        197

LENGTH_BOUNDS = {
  "aka": {
    "min_new_tokens": 25,
    "max_new_tokens": 210
  },
  "amh": {
    "min_new_tokens": 8,
    "max_new_tokens": 60
  },
  "eng": {
    "min_new_tokens": 20,
    "max_new_tokens": 151
  },
  "lug": {
    "min_new_tokens": 16,
    "max_new_tokens": 180
  },
  "swa": {
    "min_new_tokens": 20,
    "max_new_tokens": 197
  }
}
✅ Step 2 complete.


In [ ]:
# %% — Cell 5: STEP 3 — Carve Held-Out Slice
print("\n" + "=" * 70)
print("STEP 3 — CARVE HELD-OUT SLICE FROM TRAIN (5%)")
print("=" * 70)

train_core, held_out = train_test_split(train, test_size=0.05, stratify=train[lang_col], random_state=42)
train_core = train_core.reset_index(drop=True)
held_out = held_out.reset_index(drop=True)
train_core.to_csv(PROCESSED_DIR/"train_core.csv", index=False, encoding="utf-8")
held_out.to_csv(PROCESSED_DIR/"held_out.csv", index=False, encoding="utf-8")

print("train_core per-language counts:")
print(lang_distribution(train_core, lang_col).to_string())
print("\nheld_out per-language counts:")
print(lang_distribution(held_out, lang_col).to_string())

print("\n╔══════════════════════════════════════════════════════════╗")
print("║  held_out.csv is now SEALED.                             ║")
print("║  Val.csv is our local leaderboard proxy.                 ║")
print("║  train_core.csv is everything available for training.    ║")
print("║  Do NOT load held_out.csv again until final week.        ║")
print("╚══════════════════════════════════════════════════════════╝")
print("✅ Step 3 complete.")


STEP 3 — CARVE HELD-OUT SLICE FROM TRAIN (5%)
train_core per-language counts:
         count    pct
subset               
Aka_Gha   4232  14.94
Amh_Eth   1748   6.17
Eng_Eth   3719  13.13
Eng_Gha   4221  14.91
Eng_Ken   1976   6.98
Eng_Uga   7243  25.58
Lug_Uga   3214  11.35
Swa_Ken   1966   6.94

held_out per-language counts:
         count    pct
subset               
Aka_Gha    223  14.96
Amh_Eth     92   6.17
Eng_Eth    196  13.15
Eng_Gha    222  14.89
Eng_Ken    104   6.98
Eng_Uga    381  25.55
Lug_Uga    169  11.33
Swa_Ken    104   6.98

╔══════════════════════════════════════════════════════════╗
║  held_out.csv is now SEALED.                             ║
║  Val.csv is our local leaderboard proxy.                 ║
║  train_core.csv is everything available for training.    ║
║  Do NOT load held_out.csv again until final week.        ║
╚══════════════════════════════════════════════════════════╝
✅ Step 3 complete.


In [ ]:
# %% — Cell 6: STEP 4 — Build and Calibrate Evaluation Pipeline
print("\n" + "=" * 70)
print("STEP 4 — BUILD AND CALIBRATE LOCAL EVALUATION PIPELINE")
print("=" * 70)

from rouge_score import rouge_scorer
import torch
import torch.nn.functional as F

# 4a. ROUGE scorer with Unicode whitespace tokenizer
class UnicodeWhitespaceTokenizer:
    def tokenize(self, text): return re.findall(r"\S+", str(text), flags=re.UNICODE)

def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rouge1","rougeL"], use_stemmer=False,
                                       tokenizer=UnicodeWhitespaceTokenizer())
    r1s, rls = [], []
    for pred, ref in zip(predictions, references):
        s = scorer.score(ref, pred)
        r1s.append(s["rouge1"].fmeasure); rls.append(s["rougeL"].fmeasure)
    return {"rouge1": np.mean(r1s), "rougeL": np.mean(rls),
            "rouge1_per_row": r1s, "rougeL_per_row": rls}

# 4b. AfroLM BertScore (token-level proper P/R/F1)
AFROLM_MODEL_ID = "bonadossou/afrolm_active_learning"
AFROLM_FALLBACK = "intfloat/multilingual-e5-base"
_afrolm_bundle = None
_afrolm_warning = None

def load_afrolm():
    global _afrolm_bundle, _afrolm_warning
    if _afrolm_bundle is not None: return _afrolm_bundle
    from transformers import AutoModel, AutoTokenizer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    for mid in [AFROLM_MODEL_ID, AFROLM_FALLBACK]:
        try:
            tok = AutoTokenizer.from_pretrained(mid)
            mdl = AutoModel.from_pretrained(mid).to(device).eval()
            if mid != AFROLM_MODEL_ID:
                _afrolm_warning = f"⚠️ PROXY METRIC: using {mid} instead of AfroLM."
                print(_afrolm_warning)
            _afrolm_bundle = (tok, mdl, device, mid)
            print(f"Loaded AfroLM encoder: {mid}")
            return _afrolm_bundle
        except Exception as e:
            print(f"  Failed to load {mid}: {e}")
    raise RuntimeError("Cannot load any AfroLM encoder")

def embed_tokens(text, afrolm=None):
    """Return L2-normed token embeddings (n_tokens, dim) with CLS/SEP stripped."""
    if afrolm is None: afrolm = load_afrolm()
    tok, mdl, device, _ = afrolm
    max_len = min(256, int(getattr(mdl.config, "max_position_embeddings", 256) or 256) - 2)
    enc = tok(text, return_tensors="pt", truncation=True, max_length=max(8, max_len))
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = mdl(**enc)
    h = out.last_hidden_state.squeeze(0).cpu().numpy()
    mask = enc["attention_mask"].squeeze(0).cpu().numpy().astype(bool)
    h = h[mask]
    if h.shape[0] >= 3: h = h[1:-1]  # Strip CLS/SEP
    norms = np.clip(np.linalg.norm(h, axis=1, keepdims=True), 1e-8, None)
    return h / norms

def pair_bertscore(pred_emb, ref_emb):
    if pred_emb.size == 0 or ref_emb.size == 0: return 0.0, 0.0, 0.0
    sim = pred_emb @ ref_emb.T
    p = float(sim.max(axis=1).mean()); r = float(sim.max(axis=0).mean())
    if p + r == 0: return 0.0, 0.0, 0.0
    return p, r, 2*p*r/(p+r)

def afrolm_bertscore_batch(predictions, references):
    """Compute token-level BertScore F1 per row."""
    afrolm = load_afrolm()
    f1s = []
    for pred, ref in zip(predictions, references):
        pe = embed_tokens(str(pred), afrolm)
        re_ = embed_tokens(str(ref), afrolm)
        _, _, f = pair_bertscore(pe, re_)
        f1s.append(f)
    return {"afrolm_bs": np.mean(f1s), "afrolm_bs_per_row": f1s}

# 4c. LLM Judge — SKIPPED on free GPU to save memory
# Using 3-metric proxy formula instead: R1*0.3125 + RL*0.3125 + AfroLM*0.375
JUDGE_AVAILABLE = False
print("⚠️ LLM Judge SKIPPED (free GPU memory conservation).")
print("  Using 3-metric proxy: R1×0.3125 + RL×0.3125 + AfroLM×0.375")

# 4d. Combined score functions
def combined_score_4(r1, rl, bs, judge): return 0.25*r1 + 0.25*rl + 0.30*bs + 0.20*judge
def combined_score_proxy(r1, rl, bs): return 0.3125*r1 + 0.3125*rl + 0.375*bs

def full_eval_report(predictions, val_df):
    refs = val_df[answer_col].astype(str).tolist()
    langs = [LANG_CODE_MAP[str(l)] for l in val_df[lang_col].astype(str).tolist()]
    rouge = compute_rouge(predictions, refs)
    afrolm = afrolm_bertscore_batch(predictions, refs)
    combined = combined_score_proxy(rouge["rouge1"], rouge["rougeL"], afrolm["afrolm_bs"])
    # Per-row combined
    combined_per_row = [combined_score_proxy(rouge["rouge1_per_row"][i], rouge["rougeL_per_row"][i],
                                             afrolm["afrolm_bs_per_row"][i]) for i in range(len(predictions))]
    # Per-language breakdown
    per_lang = {}
    for lang in sorted(set(langs)):
        mask = [i for i,l in enumerate(langs) if l==lang]
        r1l = np.mean([rouge["rouge1_per_row"][i] for i in mask])
        rll = np.mean([rouge["rougeL_per_row"][i] for i in mask])
        bsl = np.mean([afrolm["afrolm_bs_per_row"][i] for i in mask])
        per_lang[lang] = {"combined": combined_score_proxy(r1l,rll,bsl),
                          "rouge1":r1l, "rougeL":rll, "afrolm_bs":bsl, "judge":None}
    return {"combined":combined, "r1":rouge["rouge1"], "rl":rouge["rougeL"],
            "afrolm_bs":afrolm["afrolm_bs"], "judge":None, "per_language":per_lang,
            "scoring_mode":"proxy_3_metric",
            "rouge1_per_row":rouge["rouge1_per_row"], "rougeL_per_row":rouge["rougeL_per_row"],
            "afrolm_bs_per_row":afrolm["afrolm_bs_per_row"], "judge_per_row":[None]*len(predictions),
            "combined_per_row":combined_per_row}

# 4e. Calibration checks
print("\nRunning calibration checks...")
cal_sample = val.sample(n=min(10, len(val)), random_state=SEED).reset_index(drop=True)

# Check A: perfect predictions
perfect_preds = cal_sample[answer_col].astype(str).tolist()
perfect_report = full_eval_report(perfect_preds, cal_sample)

# Check B: wrong-language predictions
random.seed(SEED)
full_langs = [LANG_CODE_MAP[str(v)] for v in val[lang_col].astype(str).tolist()]
full_refs = val[answer_col].astype(str).tolist()
sample_langs = [LANG_CODE_MAP[str(v)] for v in cal_sample[lang_col].astype(str).tolist()]
wrong_preds = []
for lang in sample_langs:
    candidates = [r for r, l in zip(full_refs, full_langs) if l != lang]
    wrong_preds.append(random.choice(candidates))
wrong_report = full_eval_report(wrong_preds, cal_sample)

check_a = perfect_report["combined"]
check_b = wrong_report["combined"]
check_a_pass = check_a >= 0.95
check_b_pass = check_b <= 0.20

print("\nCALIBRATION CHECK")
print("─" * 55)
print(f"Check A (prediction = reference) : {check_a:.4f}  [{'PASS ≥ 0.95' if check_a_pass else 'FAIL'}]")
print(f"Check B (wrong language shuffle)  : {check_b:.4f}  [{'PASS ≤ 0.20' if check_b_pass else 'FAIL'}]")
print("─" * 55)

if not check_a_pass:
    print("⚠️ Check A failed — but with proxy scoring (no judge), threshold is approximate.")
    print("  Proceeding with caution if score > 0.85...")
    if check_a < 0.85:
        raise RuntimeError("Calibration Check A critically failed. Fix evaluation pipeline.")

if not check_b_pass:
    print("⚠️ Check B failed — wrong-language scores too high.")
    if check_b > 0.35:
        raise RuntimeError("Calibration Check B critically failed. AfroLM BertScore is broken.")

# Unload AfroLM to free memory for generation model
del _afrolm_bundle
_afrolm_bundle = None
cleanup_gpu()
print("\n✅ Step 4 complete. AfroLM unloaded to free GPU memory for generation.")



STEP 4 — BUILD AND CALIBRATE LOCAL EVALUATION PIPELINE
⚠️ LLM Judge SKIPPED (free GPU memory conservation).
  Using 3-metric proxy: R1×0.3125 + RL×0.3125 + AfroLM×0.375

Running calibration checks...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/755 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/6.01M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.06G [00:00<?, ?B/s]

XLMRobertaModel LOAD REPORT from: bonadossou/afrolm_active_learning
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded AfroLM encoder: bonadossou/afrolm_active_learning

CALIBRATION CHECK
───────────────────────────────────────────────────────
Check A (prediction = reference) : 1.0000  [PASS ≥ 0.95]
Check B (wrong language shuffle)  : 0.1536  [PASS ≤ 0.20]
───────────────────────────────────────────────────────

✅ Step 4 complete. AfroLM unloaded to free GPU memory for generation.


In [ ]:
from huggingface_hub import login
import os; login(token=os.environ["HF_TOKEN"])


In [ ]:
# %% [markdown]
# # AfroHealth QA — exp001 Full Pipeline (Part 2: Steps 5-8)
# **Continue from Part 1** — all variables from Part 1 must be in scope

# %% — Cell 7: STEP 5a — Load Generation Model
print("\n" + "=" * 70)
print("STEP 5 — GENERATE PREDICTIONS: AYA-EXPANSE-8B ZERO-SHOT")
print("=" * 70)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.auto import tqdm

# ── Prompt templates with explicit "answer in this language" instructions ──
# The system instruction forces the model to NEVER refuse and ALWAYS respond
# in the same language as the question.
SYSTEM_INSTRUCTIONS = {
    "swa": "Wewe ni daktari mtaalamu. Jibu swali la afya hapa chini kwa Kiswahili. Toa jibu kamili na la moja kwa moja. Usiseme huwezi kujibu.",
    "lug": "Oli omusawo omukugu. Ddamu ekibuuzo kino eky'obulamu mu Luganda. Wa eddamu erijjuvu era ery'obwesigwa. Togamba nti toyinza kuddamu.",
    "aka": "Woyɛ dɔkota nimdeɛ. Bua afutuo asɛmmisa yi so wɔ Akan kasa mu. Ma mmuaeɛ a ɛyɛ pɛpɛɛpɛ. Nnka sɛ wontumi mmua so.",
    "amh": "እርስዎ ልምድ ያለው ሐኪም ነዎት። ይህን የጤና ጥያቄ በአማርኛ ይመልሱ። ሙሉ እና ቀጥተኛ መልስ ይስጡ። መመለስ አልችልም አይበሉ።",
    "eng": "You are an expert doctor. Answer the following health question in English. Give a complete and direct answer. Never refuse to answer.",
}

PROMPTS = {
    "swa": "Swali: {question}\n\nJibu:",
    "lug": "Ekibuuzo: {question}\n\nEky'okuddamu:",
    "aka": "Asɛmmisa: {question}\n\nMmuaeɛ:",
    "amh": "ጥያቄ: {question}\n\nመልስ:",
    "eng": "Question: {question}\n\nAnswer:",
}
ANSWER_MARKERS = {
    "swa": "Jibu:", "lug": "Eky'okuddamu:", "aka": "Mmuaeɛ:",
    "amh": "መልስ:", "eng": "Answer:",
}

# ── Refusal detection patterns ──
# If the model outputs any of these, we treat it as a refusal and retry.
REFUSAL_PATTERNS = [
    "i apologize", "i'm sorry", "i cannot", "i can't",
    "i don't have enough context", "i'm not able to",
    "i am not able to", "i do not have enough",
    "it appears to be in", "the text appears",
    "i'm unable to", "i am unable to",
    "as an ai", "as a language model",
    "i don't understand", "i do not understand",
    "sensitive nature", "i cannot provide",
    "not able to fully understand",
]

def is_refusal(text):
    """Returns True if the generated text matches a known refusal pattern."""
    lower = text.lower().strip()
    # Very short answers are suspicious too
    if len(lower) < 5:
        return True
    return any(pat in lower for pat in REFUSAL_PATTERNS)

def strip_prompt_artefacts(text, lang):
    marker = ANSWER_MARKERS.get(lang, "")
    if marker and marker in text:
        text = text.split(marker, 1)[-1]
    text = text.strip()
    if "\n\n" in text:
        text = text.split("\n\n", 1)[0].strip()
    return text

def build_prompt(question, lang, tokenizer, force_answer=False):
    """Build a prompt with system instructions.
    If force_answer=True, uses an even more forceful prompt for retry."""
    sys_msg = SYSTEM_INSTRUCTIONS[lang]
    user_content = PROMPTS[lang].format(question=question)

    if force_answer:
        # On retry: even more explicit instruction
        force_suffix = {
            "swa": " Jibu moja kwa moja bila kusita:",
            "lug": " Ddamu ku biseera bino awatali kwebuuza:",
            "aka": " Bua ntɛm ara:",
            "amh": " አሁን በቀጥታ ይመልሱ:",
            "eng": " Answer directly now:",
        }
        user_content = user_content + force_suffix.get(lang, " Answer directly:")

    if getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": user_content},
        ]
        try:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            # Some models don't support system role — merge into user msg
            merged = sys_msg + "\n\n" + user_content
            messages = [{"role": "user", "content": merged}]
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
    return sys_msg + "\n\n" + user_content

# Load model with OOM fallback chain
# Priority: AfriqueLlama is purpose-built for 20 African languages (incl.
# Swahili, Amharic). Aya/Qwen are general multilingual fallbacks.
MODEL_CHAIN = [
    "McGill-NLP/AfriqueLlama-8B",       # Purpose-built for 20 African languages
    "CohereForAI/aya-expanse-8b",        # Broad multilingual fallback
    "Qwen/Qwen2.5-7B-Instruct",         # General multilingual
]

assert torch.cuda.is_available(), "No CUDA GPU available! Cannot proceed with generation."

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

gen_model_id = None
gen_tokenizer = None
gen_model = None
for mid in MODEL_CHAIN:
    try:
        print(f"Attempting to load {mid}...")
        gen_tokenizer = AutoTokenizer.from_pretrained(mid)
        gen_model = AutoModelForCausalLM.from_pretrained(
            mid, quantization_config=bnb_config,
            device_map="auto", trust_remote_code=True,
        )
        gen_model.eval()
        gen_model_id = mid
        print(f"✅ Loaded: {mid}")
        break
    except Exception as e:
        print(f"  Failed: {e}")

if gen_model is None:
    raise RuntimeError("Could not load any generation model!")



STEP 5 — GENERATE PREDICTIONS: AYA-EXPANSE-8B ZERO-SHOT
Attempting to load McGill-NLP/AfriqueLlama-8B...


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

✅ Loaded: McGill-NLP/AfriqueLlama-8B


In [ ]:
# gc.collect()
# torch.cuda.empty_cache()

# gen_model_id = "Qwen/Qwen2.5-3B-Instruct"
# print(f"Loading {gen_model_id}...")
# gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_id)
# gen_model = AutoModelForCausalLM.from_pretrained(
#     gen_model_id,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
# )
# gen_model.eval()
# print(f"✅ Loaded: {gen_model_id}")


In [ ]:
# # Unload old model
# try:
#     del gen_model, gen_tokenizer
# except:
#     pass
# gc.collect()
# torch.cuda.empty_cache()

# # Load 1.5B model in FP16 — no bitsandbytes, fits T4 easily (~3GB)
# gen_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
# print(f"Loading {gen_model_id} in FP16...")
# gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_id)
# gen_model = AutoModelForCausalLM.from_pretrained(
#     gen_model_id,
#     dtype=torch.float16,
#     device_map="auto",
#     trust_remote_code=True,
# )
# gen_model.eval()
# print(f"✅ Loaded: {gen_model_id}")
# print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")


In [ ]:
# The original LENGTH_BOUNDS allowed up to 210 tokens per answer (based on
# training data p90 statistics). On a free-tier GPU this makes generation
# extremely slow because the model must run ~200 autoregressive forward passes
# per row. Capping at 80 tokens cuts generation time by ~60% while still
# producing answers long enough to be useful. We can restore the full bounds
# in exp002 when using a faster setup or a longer GPU session.
LENGTH_BOUNDS = {k: {"min_new_tokens": 1, "max_new_tokens": 80} for k in LENGTH_BOUNDS}
print(f"Capped LENGTH_BOUNDS: {LENGTH_BOUNDS}")


Capped LENGTH_BOUNDS: {'aka': {'min_new_tokens': 1, 'max_new_tokens': 80}, 'amh': {'min_new_tokens': 1, 'max_new_tokens': 80}, 'eng': {'min_new_tokens': 1, 'max_new_tokens': 80}, 'lug': {'min_new_tokens': 1, 'max_new_tokens': 80}, 'swa': {'min_new_tokens': 1, 'max_new_tokens': 80}}


In [ ]:
# %% — Cell 8: STEP 5b — Generate Val Predictions & Score
print("\n--- Generating Val predictions ---")

def single_generate(prompt, tokenizer, model, device, bounds, use_sampling=False):
    """Run a single generation call. Returns raw decoded text."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    gen_kwargs = {
        "min_new_tokens": bounds["min_new_tokens"],
        "max_new_tokens": min(bounds["max_new_tokens"], 256),  # Cap for speed
        "no_repeat_ngram_size": 3,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if use_sampling:
        # Sampling can help escape refusal patterns on retry
        gen_kwargs.update({"do_sample": True, "temperature": 0.7, "top_p": 0.9})
    else:
        # Greedy decoding — ~3x faster than beam search, fits within
        # Kaggle's 12h GPU limit for the full 6334-row run.
        gen_kwargs.update({"num_beams": 1, "do_sample": False})

    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def generate_answers(df, split_name, tokenizer, model, length_bounds):
    """Generate answers with refusal detection, retry, and checkpointing.

    Strategy:
    1. Generate with beam search + system instructions
    2. If refusal detected → retry with force_answer=True + sampling
    3. If still refused → use the question itself as a minimal fallback
       (echoing the question scores higher than an English refusal)
    """
    predictions = []
    refusal_count = 0
    checkpoint_path = PROCESSED_DIR / f"predictions_{split_name}_partial.csv"
    start_idx = 0
    if checkpoint_path.exists():
        partial = pd.read_csv(checkpoint_path, dtype=str).fillna("")
        predictions = partial["prediction"].tolist()
        start_idx = len(predictions)
        print(f"Resuming from checkpoint at row {start_idx}")

    device = next(model.parameters()).device

    for i in tqdm(range(start_idx, len(df)), desc=f"Generating [{split_name}]"):
        row = df.iloc[i]
        lang = canonical_prompt_language(LANG_CODE_MAP[str(row[lang_col])])
        q = str(row[question_col])
        bounds = length_bounds[lang]

        # ── Attempt 1: beam search with system instructions ──
        prompt = build_prompt(q, lang, tokenizer, force_answer=False)
        answer = single_generate(prompt, tokenizer, model, device, bounds, use_sampling=False)
        answer = strip_prompt_artefacts(answer, lang)

        # ── Attempt 2: if refusal detected, retry with forced prompt + sampling ──
        if is_refusal(answer):
            prompt_forced = build_prompt(q, lang, tokenizer, force_answer=True)
            answer = single_generate(prompt_forced, tokenizer, model, device, bounds, use_sampling=True)
            answer = strip_prompt_artefacts(answer, lang)

        # ── Attempt 3: if STILL refusing, echo the question as fallback ──
        # Rationale: an in-language echo scores higher on ROUGE/BertScore
        # than an English refusal message.
        if is_refusal(answer):
            answer = q  # The question itself, already in the target language
            refusal_count += 1

        predictions.append(answer)

        # Checkpoint every 10 rows
        if (i + 1) % 10 == 0:
            pd.DataFrame({
                "id": df[id_col].astype(str).tolist()[:len(predictions)],
                "prediction": predictions
            }).to_csv(checkpoint_path, index=False, encoding="utf-8")
            # Print refusal stats periodically
            if refusal_count > 0:
                print(f"  ⚠️ Refusals so far: {refusal_count}/{len(predictions)} "
                      f"({100*refusal_count/len(predictions):.1f}%)")

    # Final save
    pd.DataFrame({
        "id": df[id_col].astype(str).tolist()[:len(predictions)],
        "prediction": predictions
    }).to_csv(checkpoint_path, index=False, encoding="utf-8")

    print(f"\n📊 Refusal stats for [{split_name}]: {refusal_count}/{len(predictions)} "
          f"({100*refusal_count/len(predictions):.1f}%) fell back to question-echo")
    return predictions

val_predictions = generate_answers(val, "val", gen_tokenizer, gen_model, LENGTH_BOUNDS)
pd.DataFrame({id_col: val[id_col].astype(str), "prediction": val_predictions}
             ).to_csv(PROCESSED_DIR/"val_predictions_raw.csv", index=False, encoding="utf-8")



--- Generating Val predictions ---


Generating [val]:   0%|          | 0/6684 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:

# %% — Cell 9: STEP 5f — Score on Val (mandatory gate)
print("\n--- Scoring Val predictions ---")
# Reload AfroLM for scoring (it was unloaded to free memory)
_afrolm_bundle = None  # Reset
val_report = full_eval_report(val_predictions, val)

print("\n" + "═" * 62)
print("  LOCAL VAL SCORE (Zindi's official val set — best LB proxy)")
print("═" * 62)
print(f"  Combined score : {val_report['combined']:.6f}")
print(f"    ROUGE-1      : {val_report['r1']:.6f}  (weight 0.25)")
print(f"    ROUGE-L      : {val_report['rl']:.6f}  (weight 0.25)")
print(f"    AfroLM-BS    : {val_report['afrolm_bs']:.6f}  (weight 0.30)")
judge_str = "SKIPPED (proxy)" if val_report["judge"] is None else f"{val_report['judge']:.6f}"
print(f"    LLM-Judge    : {judge_str}  (weight 0.20)")
print()
print("  Per-language combined:")
for lang in sorted(val_report["per_language"]):
    print(f"    {lang} : {val_report['per_language'][lang]['combined']:.6f}")
print()
print(f"  Leaderboard #1  : {TARGET_LEADERBOARD}")
delta = val_report["combined"] - TARGET_LEADERBOARD
status = "WITHIN ±0.02" if abs(delta)<=0.02 else ("ABOVE" if delta>0 else "BELOW")
print(f"  Our val score   : {val_report['combined']:.6f}  →  [{status}]")
print("═" * 62)

# Go / No-Go gate
if val_report["combined"] < 0.50:
    raise RuntimeError("STOP: combined val score < 0.50. Pipeline is broken.")
if val_report["combined"] >= 0.75:
    print("\n🎉 Local Val score is already above the current public leaderboard leader!")


In [ ]:
# %% — Cell 10: STEP 5g — Generate Test Predictions
print("\n--- Generating Test predictions ---")
# Unload AfroLM again, keep generation model
del _afrolm_bundle
_afrolm_bundle = None
cleanup_gpu()

test_predictions = generate_answers(test, "test", gen_tokenizer, gen_model, LENGTH_BOUNDS)
pd.DataFrame({id_col: test[id_col].astype(str), "prediction": test_predictions}
             ).to_csv(PROCESSED_DIR/"test_predictions_raw.csv", index=False, encoding="utf-8")
print(f"✅ Generated {len(test_predictions)} test predictions.")

# Unload generation model — no longer needed
del gen_model, gen_tokenizer
cleanup_gpu()
print("Generation model unloaded.")


In [ ]:

# %% — Cell 11: STEP 6 — Build, Validate, and Save Submission
print("\n" + "=" * 70)
print("STEP 6 — BUILD, VALIDATE, AND SAVE THE SUBMISSION FILE")
print("=" * 70)

# 6a. Read SampleSubmission format
print(f"SampleSubmission columns: {sample.columns.tolist()}")
print(f"SampleSubmission shape: {sample.shape}")
sub_columns = tuple(sample.columns.tolist())

# 6b. Build submission
submission = pd.DataFrame({sub_columns[0]: test[id_col].astype(str).values})
for tc in sub_columns[1:]:
    submission[tc] = test_predictions
submission = submission[list(sub_columns)]

# 6c. 14-point validation
print("\nSUBMISSION VALIDATION CHECKLIST")
print("─" * 60)
checks = []
checks.append(("Column count matches", submission.shape[1] == sample.shape[1]))
checks.append(("Column names match", submission.columns.tolist() == sample.columns.tolist()))
checks.append(("Column order matches", list(submission.columns) == list(sample.columns)))
checks.append(("No extra/index columns", "Unnamed: 0" not in submission.columns))

# All target columns identical
n_check = min(100, len(submission))
targets_identical = all(submission.iloc[:n_check, i].equals(submission.iloc[:n_check, 1])
                        for i in range(2, submission.shape[1]))
checks.append(("All Target columns identical (100 rows)", targets_identical))
checks.append((f"Row count = Test.csv ({len(test)})", len(submission) == len(test)))
checks.append(("ID values match Test.csv", set(submission.iloc[:,0].astype(str)) == set(test[id_col].astype(str))))
checks.append(("ID order matches Test.csv", submission.iloc[:,0].astype(str).tolist() == test[id_col].astype(str).tolist()))
checks.append(("No null/NaN", not submission.isna().any().any()))

target_cols = list(sub_columns[1:])
checks.append(("No empty strings in Target", not (submission[target_cols] == "").any().any()))

# Whitespace-only check
ws_only = False
for tc in target_cols:
    if submission[tc].astype(str).str.strip().eq("").any():
        ws_only = True; break
checks.append(("No whitespace-only Target", not ws_only))

# Save first, then check encoding
timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
submission_path = SUBMISSIONS_DIR / f"{timestamp}_exp001_aya_zeroshot.csv"
submission.to_csv(submission_path, index=False, encoding="utf-8")

with open(submission_path, "rb") as f:
    raw_head = f.read(4)
checks.append(("UTF-8 no BOM", raw_head[:3] != b"\xef\xbb\xbf" and raw_head[:2] != b"\xff\xfe"))

max_chars = int(submission[target_cols[0]].astype(str).str.len().max())
min_chars = int(submission[target_cols[0]].astype(str).str.len().min())
checks.append(("No Target > 1000 chars", max_chars <= 1000))
checks.append(("No Target < 5 chars", min_chars >= 5))

all_passed = True
for label, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed: all_passed = False
    print(f"[{status}] {label}")

print("─" * 60)
print(f"RESULT: [{'ALL PASSED — READY TO SUBMIT' if all_passed else 'FAILED'}]")
if not all_passed:
    raise RuntimeError("Submission validation failed!")
print(f"\nSaved: {submission_path}")

# 6e. Save metadata sidecar
metadata = {
    "experiment_id": EXPERIMENT_ID,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "hypothesis": HYPOTHESIS,
    "model_id": gen_model_id,
    "decoding": "beam5, length_penalty=1.0, no_repeat_ngram_size=3",
    "augmentation": "none",
    "data_files": {"train":"Train.csv","val":"Val.csv","test":"Test.csv","sample":"SampleSubmission.csv"},
    "local_val_scores": {
        "combined": val_report["combined"], "rouge1": val_report["r1"],
        "rougeL": val_report["rl"], "afrolm_bs": val_report["afrolm_bs"],
        "judge": val_report["judge"],
        "per_language": {k: v["combined"] for k,v in val_report["per_language"].items()},
    },
    "val_rows_used": len(val), "test_rows_used": len(test),
    "submission_file": str(submission_path.name),
    "submission_row_count": len(submission),
    "submission_validation": "ALL_PASSED" if all_passed else "FAILED",
    "leaderboard_target": TARGET_LEADERBOARD,
    "public_lb_score": None,
    "notes": "First submission. Update public_lb_score after upload.",
}
sidecar = submission_path.with_suffix(submission_path.suffix + ".json")
sidecar.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Sidecar saved: {sidecar}")
print("✅ Step 6 complete.")

In [ ]:

# %% — Cell 12: STEP 7 — Experiment Log Entry
print("\n" + "=" * 70)
print("STEP 7 — EXPERIMENT LOG ENTRY")
print("=" * 70)

log_line = (
    f"| {EXPERIMENT_ID} | {datetime.now().date().isoformat()} | "
    f"Zero-shot Aya beam5 lang-native prompts, beat 0.768095 | {gen_model_id} | "
    f"none | beam5 lp=1.0 nrn=3 len-matched | {val_report['r1']:.6f} | "
    f"{val_report['rl']:.6f} | {val_report['afrolm_bs']:.6f} | "
    f"{'SKIPPED' if val_report['judge'] is None else val_report['judge']:.6f} | "
    f"{val_report['combined']:.6f} | PENDING | "
    f"{submission_path.name} | First run. Val.csv used as LB proxy. |"
)
log_path = EXPERIMENTS_DIR / "LOG.md"
log_path.write_text(f"# Experiment Log\n\n| ID | Date | Hypothesis | Model | Data | Params | R1 | RL | AfroLM-BS | Judge | Combined | LB | File | Notes |\n|---|---|---|---|---|---|---|---|---|---|---|---|---|---|\n{log_line}\n", encoding="utf-8")
print(log_line)
print("✅ Step 7 complete.")


In [ ]:
# %% — Cell 13: STEP 8 — Post-Run Diagnostics
print("\n" + "=" * 70)
print("STEP 8 — POST-RUN DIAGNOSTICS AND NEXT-STEP RECOMMENDATION")
print("=" * 70)

# 8a. Per-language weakness table
print("\nWEAKEST SUB-METRIC PER LANGUAGE")
print(f"{'lang':<6} {'ROUGE-1':>9} {'ROUGE-L':>9} {'AfroLM-BS':>11} {'WEAKEST':>10}")
print("─" * 50)
for lang in sorted(val_report["per_language"]):
    m = val_report["per_language"][lang]
    metrics = {"ROUGE-1": m["rouge1"], "ROUGE-L": m["rougeL"], "AfroLM-BS": m["afrolm_bs"]}
    weakest = min(metrics, key=metrics.get)
    print(f"{lang:<6} {m['rouge1']:>9.4f} {m['rougeL']:>9.4f} {m['afrolm_bs']:>11.4f} {weakest:>10}")

# 8b. Sample predictions
print("\n--- 3 Sample Val Predictions Per Language ---")
val_scored = val.copy().reset_index(drop=True)
val_scored["prediction"] = val_predictions
val_scored["row_score"] = val_report["combined_per_row"]
val_scored["r1_score"] = val_report["rouge1_per_row"]
val_scored["rl_score"] = val_report["rougeL_per_row"]
val_scored["bs_score"] = val_report["afrolm_bs_per_row"]
val_scored["canon_lang"] = val_scored[lang_col].astype(str).map(lambda x: LANG_CODE_MAP[str(x)])

for lang in sorted(val_scored["canon_lang"].unique()):
    examples = val_scored[val_scored["canon_lang"]==lang].head(3)
    for _, row in examples.iterrows():
        print(f"\n[{lang} — row score: {row['row_score']:.2f}]")
        print(f"Q   : {str(row[question_col])[:200]}")
        print(f"REF : {str(row[answer_col])[:200]}")
        print(f"GEN : {str(row['prediction'])[:200]}")
        print(f"Gap : R1={row['r1_score']:.2f} RL={row['rl_score']:.2f} BS={row['bs_score']:.2f}")

# 8c. exp002 recommendation
r1 = val_report["r1"]; rl = val_report["rl"]; bs = val_report["afrolm_bs"]
per_lang_scores = [v["combined"] for v in val_report["per_language"].values()]

if r1 < 0.60 and rl < 0.60:
    exp002 = "QLoRA fine-tune on train_core.csv, 3 epochs, r=32, alpha=64."
    hyp = "Fine-tuning Aya on train_core will improve lexical overlap over zero-shot."
elif bs < 0.55:
    exp002 = "Add 2 few-shot examples per language to the prompt."
    hyp = "In-language exemplars will improve semantic similarity without fine-tuning."
elif val_report["combined"] >= 0.78:
    exp002 = "Submit immediately, then fine-tune as exp002."
    hyp = "Fine-tuned model can push beyond already-leading zero-shot."
elif val_report["combined"] >= 0.75:
    exp002 = "Try length_penalty=0.8, num_beams=8, AfroLM reranking."
    hyp = "Broader beam with lighter penalty and reranking will outperform plain beam5."
elif max(per_lang_scores) - min(per_lang_scores) > 0.12:
    exp002 = "Augment the weakest language with external corpus."
    hyp = "Targeted augmentation will close the per-language score gap."
else:
    exp002 = "QLoRA fine-tune on train_core.csv, 3 epochs, r=32, alpha=64."
    hyp = "Supervised fine-tuning will outperform zero-shot baseline."

print(f"\n\nexp002 recommendation: {exp002}")
print(f"exp002 hypothesis: {hyp}")

# 8d. Final submission readiness verdict
print("\n" + "═" * 64)
print(f"  SUBMISSION READY TO UPLOAD: YES")
print("═" * 64)
print(f"  File   : {submission_path.name}")
print(f"  Rows   : {len(submission)}  (must match Test.csv: {len(test)})")
print(f"  Checks : ALL 14 PASSED")
print()
print(f"  Val score (our LB proxy) : {val_report['combined']:.6f}")
print(f"  Leaderboard target       : {TARGET_LEADERBOARD}")
target_status = "ABOVE TARGET ✓" if val_report["combined"] > TARGET_LEADERBOARD else "BELOW TARGET — exp002 plan above"
print(f"  Status                   : {target_status}")
print()
print("  NEXT STEPS:")
print("  1. Download the submission CSV from this notebook")
print("  2. Upload to Zindi: My Submissions → Upload → select the file")
print("  3. When the public score comes back, paste it to plan exp002")
print()
print("  Submissions used : 1 / 50")
print("  Budget remaining : 49")
print("═" * 64)

In [ ]:
# %% — Cell 14: Download Results (Kaggle)
# On Kaggle, files in /kaggle/working/ are automatically saved as output.
# You can download them from the notebook output tab.
print(f"\n📥 Submission file ready for download: {submission_path}")
print(f"📥 Metadata sidecar: {sidecar}")
print(f"📥 Val predictions: {PROCESSED_DIR/'val_predictions_raw.csv'}")
print(f"📥 Test predictions: {PROCESSED_DIR/'test_predictions_raw.csv'}")

# For Colab, uncomment:
# from google.colab import files
# files.download(str(submission_path))